# Data fundamentals for AI application development

**Hands-on lab | Oracle AI Database 26ai**

---

In this notebook you will:

1. Connect to Oracle AI Database 26ai and explore the Prism smart city dataset
2. Generate vector embeddings using an in-database ONNX model
3. Create an HNSW vector index and perform semantic search
4. Execute a single SQL query that combines relational JOINs, JSON dot notation, graph traversal, and vector search

**Estimated time:** 30 minutes (core) + 15 minutes (optional Hybrid Vector Search section)

**Prerequisites:** A running Oracle AI Database 26ai Free container with the Prism schema pre-loaded.

## The Prism dataset

Prism uses a curated smart city dataset with seven districts and 28 infrastructure assets including bridges, substations, pipelines, sensors, communication towers, and more. The dataset also includes maintenance logs, inspection reports with findings, and connectivity relationships between assets. All of this data lives in a single Oracle database and is projected as relational rows, JSON documents, graph relationships, and vector embeddings, with no duplication and no synchronization overhead.

## Section 0: Configuration and readiness check

Update the values below to match your LiveLabs or local Oracle AI Database 26ai environment. The readiness check that follows verifies that the database, Prism schema objects, sample data, graph, vectorized chunks, and ONNX model are available before the lab begins.


In [ ]:
import os
from dotenv import load_dotenv

# === CONFIGURATION - UPDATE THESE VALUES IF NECESSARY ===
DB_USER     = os.getenv("DBUSER", "prism")
DB_PASSWORD = os.getenv("DBPASSWORD", "CHANGE_ME")
DB_DSN = os.getenv("DBCONNECTION") or "aidbfree:1521/FREEPDB1"
ONNX_MODEL  = os.getenv("ONNX_MODEL", "ALL_MINILM_L12_V2")

# Demo constants used throughout the lab. Change these once if you customize the seed data.
DEMO_BRIDGE         = "Harbor Bridge"
DEMO_SUBSTATION     = "Substation Gamma"
DEMO_PIPELINE       = "Pipeline North-7"
DEMO_CONTROL_CENTER = "City Operations Control Center"
DEMO_EOC            = "Emergency Operations Center"

SCENARIOS = [
    {
        "name": "Bridge structural risk",
        "target_asset": DEMO_BRIDGE,
        "question": f"What evidence suggests structural risk on {DEMO_BRIDGE}?",
        "related_assets": ["Harbor Bridge Sensor Array A", "Harbor Bridge Sensor Array B", DEMO_EOC],
    },
    {
        "name": "Substation cascading outage",
        "target_asset": DEMO_SUBSTATION,
        "question": f"What recent evidence points to electrical instability at {DEMO_SUBSTATION} and what connected assets may be affected?",
        "related_assets": [DEMO_CONTROL_CENTER, DEMO_EOC, "Ironworks Water Treatment Plant", "Substation Epsilon"],
    },
    {
        "name": "Pipeline leak response",
        "target_asset": DEMO_PIPELINE,
        "question": f"What leak or pressure evidence affects {DEMO_PIPELINE} and what response actions are required?",
        "related_assets": [DEMO_CONTROL_CENTER, DEMO_EOC, "Northern Reservoir"],
    },
]

print(f"Configuration complete: {len(SCENARIOS)} scenarios configured; default demo bridge is {DEMO_BRIDGE}.")


In [ ]:
# Install and import dependencies
import oracledb
import json
import html
from IPython.display import HTML, display
from decimal import Decimal

import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Return LOB columns as Python strings instead of LOB objects
oracledb.defaults.fetch_lobs = False

def ok(msg: str) -> None:
    """Print a green check confirmation line at the end of successful cells."""
    display(HTML(
        f"<span style='color:#1a7f37;font-weight:700'>&#10003;</span>"
        f" <span style='color:#1a7f37'>{html.escape(msg)}</span>"
    ))


def show_table(headers, rows, max_width=80):
    """Display query results as a clean HTML table."""
    style_th = "border:1px solid #ddd; padding:6px 10px; background:#f4f4f4; text-align:left;"
    style_td = f"border:1px solid #ddd; padding:6px 10px; white-space:pre-wrap; max-width:{max_width}ch; word-wrap:break-word;"
    def esc(value):
        return html.escape("" if value is None else str(value))
    h = '<table style="border-collapse:collapse; font-size:13px;">'
    h += "<tr>" + "".join(f"<th style=\"{style_th}\">{esc(c)}</th>" for c in headers) + "</tr>"
    for row in rows:
        h += "<tr>" + "".join(f"<td style=\"{style_td}\">{esc(v)}</td>" for v in row) + "</tr>"
    h += "</table>"
    display(HTML(h))

ok("Dependencies loaded.")

def _json_default(obj):
    """Handle Oracle types that json.dumps doesn't know about."""
    if isinstance(obj, Decimal):
        return float(obj)
    return str(obj)

def print_json(result):
    """Pretty-print a JSON result from Oracle."""
    if isinstance(result, str):
        result = json.loads(result)
    print(json.dumps(result, indent=2, default=_json_default))

def rename_fused_score(result):
    """Rename 'score' to 'fused_score' in hybrid search results for clarity."""
    if isinstance(result, str):
        result = json.loads(result)
    for item in result:
        if 'score' in item:
            item['fused_score'] = item.pop('score')
    return result

def show_graph(edges, node_types=None, highlight_nodes=None, title=None):
    """Visualize a network graph from (from_node, relationship, to_node) tuples."""
    G = nx.DiGraph()
    for src, rel, tgt in edges:
        G.add_edge(src, tgt, label=rel)

    # Color palette by asset type
    palette = ['#4E79A7', '#F28E2B', '#59A14F', '#76B7B2', '#EDC948',
              '#B07AA1', '#FF9DA7', '#9C755F', '#BAB0AC', '#E15759']
    type_colors = {}
    if node_types:
        for i, t in enumerate(sorted(set(node_types.values()))):
            type_colors[t] = palette[i % len(palette)]

    node_colors = []
    for node in G.nodes():
        if highlight_nodes and node in highlight_nodes:
            node_colors.append('#E15759')
        elif node_types and node in node_types:
            node_colors.append(type_colors[node_types[node]])
        else:
            node_colors.append('#4E79A7')

    fig, ax = plt.subplots(figsize=(14, 9))
    pos = nx.spring_layout(G, k=2.5, seed=42)
    node_size = 2200
    # Pass the real node size to the edge renderer so arrowheads stop at the
    # circle boundary instead of being drawn underneath the node fill.
    edge_node_size = node_size + 450
    nx.draw_networkx_nodes(G, pos, node_color=node_colors,
                           node_size=node_size, alpha=0.9, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=7, font_weight='bold', ax=ax)
    nx.draw_networkx_edges(G, pos, edge_color='#555555', arrows=True,
                           arrowstyle='-|>', arrowsize=24, width=1.8,
                           node_size=edge_node_size,
                           connectionstyle='arc3,rad=0.1', ax=ax)
    edge_labels = nx.get_edge_attributes(G, 'label')
    nx.draw_networkx_edge_labels(G, pos, edge_labels,
                                  font_size=6, font_color='#444444', ax=ax)
    if title:
        ax.set_title(title, fontsize=14, fontweight='bold', pad=20)
    if node_types:
        handles = [mpatches.Patch(color=c, label=t) for t, c in type_colors.items()]
        if highlight_nodes:
            handles.append(mpatches.Patch(color='#E15759', label='critical incident'))
        ax.legend(handles=handles, loc='upper left', fontsize=8)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

ok("Functions loaded, and cell complete!")

### Connect to the database

This first check only proves that the notebook can connect to Oracle. The next cell verifies the Prism objects and data this workshop depends on.


In [ ]:
# Connect to Oracle AI Database 26ai (thin mode - no Oracle Client needed)
conn = oracledb.connect(user=DB_USER, password=DB_PASSWORD, dsn=DB_DSN)
cursor = conn.cursor()

# Verify the connection
cursor.execute("SELECT banner FROM v$version WHERE ROWNUM = 1")
banner = cursor.fetchone()[0]
print(f"{banner}")
ok("Connected successfully!")

### Readiness check

The database should already be prepared before the workshop starts: Oracle database created, Prism schema created, sample data loaded, document chunks populated, property graph created, and the `ALL_MINILM_L12_V2` ONNX embedding model loaded.

Run this cell before continuing. It performs quick checks and reports all missing pieces at once, instead of failing one cell at a time later in the lab.


In [ ]:
# Quick workshop readiness probe. Keep this cell fast and safe to re-run.

def _scalar(sql, params=None):
    cursor.execute(sql, params or {})
    row = cursor.fetchone()
    return row[0] if row else None


def _try_scalar(label, sql, params=None):
    try:
        return _scalar(sql, params), None
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"


checks = []


def check(label, ok_flag, detail, fix, required=True):
    checks.append({
        "label": label,
        "status": "OK" if ok_flag else ("WARN" if not required else "FAIL"),
        "ok": bool(ok_flag) or not required,
        "required": required,
        "detail": detail,
        "fix": "" if ok_flag else fix,
    })


# 1. Confirm the connected schema. The notebook uses unqualified object names,
# so the connected user must own the Prism objects or have synonyms in place.
current_user, err = _try_scalar("Current user", "SELECT USER FROM DUAL")
check(
    "Connected database user",
    current_user is not None,
    f"USER={current_user}" if not err else err,
    "Re-run the connection cell after checking DB_USER, DB_PASSWORD, and DB_DSN.",
)

# 2. Required relational/vector tables.
required_tables = [
    "DISTRICTS",
    "INFRASTRUCTURE_ASSETS",
    "OPERATIONAL_PROCEDURES",
    "MAINTENANCE_LOGS",
    "INSPECTION_REPORTS",
    "INSPECTION_FINDINGS",
    "ASSET_CONNECTIONS",
    "DOCUMENT_CHUNKS",
]

try:
    placeholders = ",".join(f"'{t}'" for t in required_tables)
    cursor.execute(f"""
        SELECT table_name
        FROM user_tables
        WHERE table_name IN ({placeholders})
    """)
    found_tables = {row[0] for row in cursor.fetchall()}
    missing_tables = [t for t in required_tables if t not in found_tables]
    check(
        "Required Prism tables exist",
        not missing_tables,
        f"present={len(found_tables)}/{len(required_tables)}; missing={', '.join(missing_tables) or 'none'}",
        "Create the Prism schema objects before running this lab. See ingestion/db-startup/30-create-prism-db-objects.sh.",
    )
except Exception as exc:
    found_tables = set()
    check(
        "Required Prism tables exist",
        False,
        f"{type(exc).__name__}: {exc}",
        "Create the Prism schema objects before running this lab. See ingestion/db-startup/30-create-prism-db-objects.sh.",
    )

# 3. Row-count sanity checks. Use minimums so the check survives small dataset additions.
minimum_rows = {
    "DISTRICTS": 7,
    "INFRASTRUCTURE_ASSETS": 28,
    "MAINTENANCE_LOGS": 1,
    "INSPECTION_REPORTS": 1,
    "INSPECTION_FINDINGS": 1,
    "ASSET_CONNECTIONS": 1,
    "DOCUMENT_CHUNKS": 1,
}

for table_name, minimum in minimum_rows.items():
    if table_name not in found_tables:
        check(
            f"{table_name} has data",
            False,
            "table missing",
            "Load the Prism sample data and run the chunk/embed ingestion step before continuing.",
        )
        continue
    count, err = _try_scalar(f"Count {table_name}", f"SELECT COUNT(*) FROM {table_name}")
    check(
        f"{table_name} has data",
        count is not None and count >= minimum,
        f"count={count}; expected>={minimum}" if not err else err,
        "Load the Prism sample data. If DOCUMENT_CHUNKS is empty, run the chunk/embed ingestion step.",
    )

# 4. Objects used by later sections.
for view_name, purpose in [
    ("V_CHUNKS_UNIFIED", "semantic search examples"),
    ("V_CHUNKS_OPERATIONAL_PROCEDURES", "SOP semantic search examples"),
    ("V_OPERATIONAL_PROCEDURE_SUMMARY", "SOP JSON projection examples"),
    ("V_ASSET_SPECS_SUMMARY", "JSON specification examples"),
]:
    view_count, err = _try_scalar(
        view_name,
        "SELECT COUNT(*) FROM user_views WHERE view_name = :view_name",
        {"view_name": view_name},
    )
    check(
        f"View {view_name} exists",
        view_count == 1,
        f"count={view_count}" if not err else err,
        f"Create or refresh {view_name}; it is used by {purpose}.",
    )

graph_count, err = _try_scalar(
    "CITYPULSE_GRAPH",
    "SELECT COUNT(*) FROM user_property_graphs WHERE graph_name = 'CITYPULSE_GRAPH'",
)
check(
    "Property graph CITYPULSE_GRAPH exists",
    graph_count == 1,
    f"count={graph_count}" if not err else err,
    "Create the CITYPULSE_GRAPH property graph before continuing.",
)

# 5. Schema compatibility checks that catch notebook/data drift.
criticality_col_count, err = _try_scalar(
    "CRITICALITY column",
    """
        SELECT COUNT(*)
        FROM user_tab_columns
        WHERE table_name = 'INFRASTRUCTURE_ASSETS'
          AND column_name = 'CRITICALITY'
    """,
)
check(
    "INFRASTRUCTURE_ASSETS.CRITICALITY exists",
    criticality_col_count == 1,
    f"count={criticality_col_count}" if not err else err,
    "Re-run the schema creation script that adds the CRITICALITY column.",
)

chunk_column_count, err = _try_scalar(
    "V_CHUNKS_UNIFIED columns",
    """
        SELECT COUNT(*)
        FROM user_tab_columns
        WHERE table_name = 'V_CHUNKS_UNIFIED'
          AND column_name IN ('ASSET_NAME', 'DISTRICT_NAME', 'CRITICALITY')
    """,
)
check(
    "V_CHUNKS_UNIFIED exposes asset context",
    chunk_column_count == 3,
    f"matched_columns={chunk_column_count}/3" if not err else err,
    "Refresh the helper views so vector-search examples can show asset context.",
)

substation_key_count, err = _try_scalar(
    "Substation JSON keys",
    """
        SELECT COUNT(*)
        FROM infrastructure_assets
        WHERE asset_type = 'substation'
          AND json_exists(specifications, '$.voltageRating_kv')
          AND json_exists(specifications, '$.transformerCount')
          AND json_exists(specifications, '$.peakCapacity_mw')
          AND json_exists(specifications, '$.coolingType')
    """,
)
check(
    "Substation JSON keys match notebook examples",
    substation_key_count == 3,
    f"matching_substations={substation_key_count}/3" if not err else err,
    "Reload the latest PRISM seed data or update the JSON path examples.",
)

scenario_log_count, err = _try_scalar(
    "Scenario logs",
    """
        SELECT COUNT(*)
        FROM maintenance_logs ml
        JOIN infrastructure_assets a ON a.asset_id = ml.asset_id
        WHERE a.name = :asset_name
          AND ml.narrative LIKE '%SCADA correlation drill%'
    """,
    {"asset_name": DEMO_SUBSTATION},
)
check(
    "Golden-path incident data loaded",
    scenario_log_count and scenario_log_count >= 1,
    f"matching_logs={scenario_log_count}" if not err else err,
    "Reload ingestion/db-startup/35-load-prism-initial-data.sh, then rerun vector ingestion.",
)

# 6. Demo records used throughout the notebook.
for label, asset_name in [
    ("Demo bridge", DEMO_BRIDGE),
    ("Demo substation", DEMO_SUBSTATION),
    ("Demo pipeline", DEMO_PIPELINE),
    ("Demo operations center", DEMO_CONTROL_CENTER),
    ("Demo emergency operations center", DEMO_EOC),
]:
    asset_count, err = _try_scalar(
        asset_name,
        "SELECT COUNT(*) FROM infrastructure_assets WHERE name = :name",
        {"name": asset_name},
    )
    check(
        f"{label} exists: {asset_name}",
        asset_count == 1,
        f"count={asset_count}" if not err else err,
        "Reload the Prism sample data or update the demo asset constants in this notebook.",
    )

# 7. ONNX model exists and can produce embeddings.
model_count, err = _try_scalar(
    "ONNX model",
    """
        SELECT COUNT(*)
        FROM all_mining_models
        WHERE model_name = :model_name
    """,
    {"model_name": ONNX_MODEL.upper()},
)
check(
    f"ONNX model {ONNX_MODEL} exists",
    model_count == 1,
    f"count={model_count}" if not err else err,
    "Load the ONNX embedding model.",
)

if model_count == 1:
    dims, err = _try_scalar(
        "Embedding dimensions",
        f"""
            SELECT VECTOR_DIMS(
                VECTOR_EMBEDDING({ONNX_MODEL} USING 'readiness check' AS data)
            )
            FROM DUAL
        """,
    )
    check(
        f"ONNX model {ONNX_MODEL} can embed text",
        dims is not None and dims > 0,
        f"dimensions={dims}" if not err else err,
        "Confirm the model name and that the connected user can execute VECTOR_EMBEDDING with it.",
    )
else:
    check(
        f"ONNX model {ONNX_MODEL} can embed text",
        False,
        "skipped because model was not found",
        "Load the ONNX embedding model.",
    )

# 8. The HNSW vector index should exist before learners run vector searches.
# If it is missing, create it here so the rest of the notebook can proceed.
index_count, err = _try_scalar(
    "IDX_CHUNK_EMBEDDING",
    "SELECT COUNT(*) FROM user_indexes WHERE index_name = 'IDX_CHUNK_EMBEDDING'",
)
index_detail = f"count={index_count}" if not err else err

if index_count != 1:
    try:
        cursor.execute("""
            CREATE VECTOR INDEX idx_chunk_embedding
                ON document_chunks(embedding)
                ORGANIZATION INMEMORY NEIGHBOR GRAPH
                DISTANCE COSINE
                WITH TARGET ACCURACY 95
        """)
        index_count, err = _try_scalar(
            "IDX_CHUNK_EMBEDDING after create",
            "SELECT COUNT(*) FROM user_indexes WHERE index_name = 'IDX_CHUNK_EMBEDDING'",
        )
        index_detail = f"count={index_count}"
    except Exception as exc:
        index_count = 0
        index_detail = f"Automatic create failed: {type(exc).__name__}: {exc}"

check(
    "Existing HNSW vector index IDX_CHUNK_EMBEDDING",
    index_count == 1,
    index_detail,
    "The notebook could not create the vector index automatically. Check database vector memory and privileges.",
)

# Render the readiness summary.
show_table(
    ["Check", "Status", "Detail", "Fix if needed"],
    [[c["label"], c["status"], c["detail"], c["fix"]] for c in checks],
    max_width=95,
)

failures = [c for c in checks if c["status"] == "FAIL"]
warnings = [c for c in checks if c["status"] == "WARN"]

if failures:
    fixes = "\n".join(f"- {c['label']}: {c['fix']}" for c in failures)
    raise RuntimeError(
        f"Readiness check failed: {len(failures)} required check(s) failed.\n"
        f"Fix these before continuing:\n{fixes}"
    )

if warnings:
    print(f"Readiness passed with {len(warnings)} warning(s). You can continue.")
else:
    ok("Readiness passed. You can continue with Section 1.")


> ✅ **Checkpoint: Workshop environment is ready**
>
> **Time estimate:** 3-5 minutes.
>
> Continue when the readiness summary reports no blocking failures.
>
> Warnings are usually safe to review with the facilitator. For example, a missing vector index is acceptable here because this notebook creates or recreates it later.


---
## Section 1: Explore the Prism data model

The Prism schema is already loaded with sample data. Let's see what's here before we build on top of it.

### Exercise pattern for this lab

Some sections now include optional exercise cells marked with a `╭─ EXERCISE ─╮` banner. They are designed for learners who want to practice rather than only run the prepared demo.

Each exercise cell is safe to skip. By default it prints instructions and does not run a query. To try the exercise, edit the SQL or Python in the TODO area and set `RUN_EXERCISE = True`. If you get stuck, expand the **🔎 Reveal solution** cell immediately below the exercise and paste the full solution over the exercise cell.


In [ ]:
# List the Prism tables and their row counts
objects = [
    ('DISTRICTS', 'table'),
    ('INFRASTRUCTURE_ASSETS', 'table'),
    ('OPERATIONAL_PROCEDURES', 'table'),
    ('MAINTENANCE_LOGS', 'table'),
    ('INSPECTION_REPORTS', 'table'),
    ('INSPECTION_FINDINGS', 'table'),
    ('ASSET_CONNECTIONS', 'table'),
    ('DOCUMENT_CHUNKS', 'table'),
    ('V_ASSET_SPECS_SUMMARY', 'view'),
    ('V_OPERATIONAL_PROCEDURE_SUMMARY', 'view'),
    ('V_CHUNKS_OPERATIONAL_PROCEDURES', 'view'),
    ('V_CHUNKS_UNIFIED', 'view'),
]

rows = []
for object_name, object_type in objects:
    cursor.execute(f'SELECT COUNT(*) FROM {object_name}')
    count = cursor.fetchone()[0]
    rows.append((object_name, object_type, count))

show_table(['Object', 'Type', 'Row Count'], rows)

ok("Object list complete.")


### Data dictionary and JSON specification keys

Before writing queries, inspect the learner-facing objects and the JSON keys available for each asset type. The same specifications column stores different fields for bridges, substations, pipelines, control centers, and other infrastructure assets.


In [ ]:
# Show columns for the core tables and helper views used in this lab
objects_to_describe = [
    'DISTRICTS', 'INFRASTRUCTURE_ASSETS', 'MAINTENANCE_LOGS',
    'INSPECTION_REPORTS', 'INSPECTION_FINDINGS', 'ASSET_CONNECTIONS',
    'DOCUMENT_CHUNKS', 'V_ASSET_SPECS_SUMMARY',
    'V_OPERATIONAL_PROCEDURE_SUMMARY', 'V_CHUNKS_OPERATIONAL_PROCEDURES',
    'V_CHUNKS_UNIFIED'
]

placeholders = ','.join(f"'{name}'" for name in objects_to_describe)
cursor.execute(f"""
    SELECT table_name,
           column_id,
           column_name,
           CASE
             WHEN data_type IN ('VARCHAR2', 'CHAR') THEN data_type || '(' || data_length || ')'
             ELSE data_type
           END AS data_type,
           nullable
    FROM user_tab_columns
    WHERE table_name IN ({placeholders})
    ORDER BY table_name, column_id
""")

rows = [(r[0], r[2], r[3], r[4]) for r in cursor.fetchall()]
show_table(['Object', 'Column', 'Data Type', 'Nullable'], rows, max_width=40)

ok("Cell complete. These are the columns from the core tables and views.")


In [ ]:
# Discover the JSON specification keys actually present for each asset type
cursor.execute("""
    SELECT asset_type,
           JSON_SERIALIZE(specifications RETURNING CLOB) AS spec_json
    FROM infrastructure_assets
    ORDER BY asset_type, name
""")

spec_keys = {}
spec_counts = {}
for asset_type, spec_doc in cursor.fetchall():
    if hasattr(spec_doc, 'read'):
        spec_doc = spec_doc.read()
    if isinstance(spec_doc, bytes):
        spec_doc = spec_doc.decode('utf-8')
    specs = json.loads(spec_doc) if isinstance(spec_doc, str) else spec_doc
    spec_counts[asset_type] = spec_counts.get(asset_type, 0) + 1
    spec_keys.setdefault(asset_type, set()).update(specs.keys())

rows = [
    (asset_type, spec_counts[asset_type], ', '.join(sorted(keys)))
    for asset_type, keys in sorted(spec_keys.items())
]
show_table(['Asset Type', 'Assets', 'Specification Keys'], rows, max_width=110)

ok("Cell run complete. These are the spec keys for the JSON documents in the asset_type table.")


### Known-good scenarios

These curated scenarios give you reliable starting points for exercises. Each one has structured asset records, JSON specifications, graph relationships, maintenance logs, inspection context, and vector-searchable text.


In [ ]:
scenario_rows = [
    (
        s['name'],
        s['target_asset'],
        s['question'],
        ', '.join(s['related_assets']),
    )
    for s in SCENARIOS
]
show_table(['Scenario', 'Target Asset', 'Good Search Question', 'Related Assets'], scenario_rows, max_width=95)

ok("These are a few scenarios you can use for later parts of the lab.")


### Relational data

Infrastructure assets are stored in normalized relational tables with foreign key relationships to districts.

In [ ]:
# Peek at infrastructure assets with their district and priority signal
cursor.execute("""
    SELECT a.asset_id, a.name, a.asset_type, a.status,
           a.criticality, d.name AS district
    FROM infrastructure_assets a
    JOIN districts d ON d.district_id = a.district_id
    ORDER BY a.asset_id
    FETCH FIRST 10 ROWS ONLY
""")

cols = [c[0] for c in cursor.description]
show_table(cols, cursor.fetchall())

ok("These are infrastructure assets with their district and priority.")


> ✅ **Checkpoint: Relational asset data is working**
>
> You successfully queried the core `INFRASTRUCTURE_ASSETS` table with standard SQL.
>
> **Expected result:** rows for smart-city assets, including familiar demo assets such as `Harbor Bridge` or `Substation Gamma`.


### Criticality-driven triage

Criticality is a 1-5 operational priority score. It gives later graph and vector-search results a simple business signal: high-criticality assets deserve earlier attention when evidence is otherwise similar.


In [ ]:
# Sort assets by operational importance before looking at incidents
cursor.execute("""
    SELECT a.name,
           a.asset_type,
           a.criticality,
           d.name AS district,
           a.status
    FROM infrastructure_assets a
    JOIN districts d ON d.district_id = a.district_id
    ORDER BY a.criticality DESC, a.asset_type, a.name
    FETCH FIRST 12 ROWS ONLY
""")

cols = [c[0] for c in cursor.description]
show_table(cols, cursor.fetchall())

ok("This is the list of assets by operational importance.")


In [ ]:
# ╭─ EXERCISE §1.A ─────────────────────────────────────────────────────╮
# │ TASK: inspect a different slice of infrastructure assets.           │
# │                                                                     │
# │ HOW TO COMPLETE THIS CELL:                                          │
# │   1. Replace the TODO SQL with a query that returns asset name,     │
# │      asset_type, status, district, and criticality.                 │
# │   2. Filter to one district or one asset type.                      │
# │   EXPECTED OUTPUT: name, asset_type, status, district, criticality. │
# │   3. Set RUN_EXERCISE = True and run the cell.                      │
# │                                                                     │
# │ Stuck? Expand the 🔎 solution cell below and paste the full answer. │
# ╰─────────────────────────────────────────────────────────────────────╯

RUN_EXERCISE = False

exercise_sql = """
    -- TODO: write a SELECT against infrastructure_assets joined to districts.
    -- Hint: filter by d.name or a.asset_type.
"""

if RUN_EXERCISE:
    cursor.execute(exercise_sql)
    cols = [c[0] for c in cursor.description]
    show_table(cols, cursor.fetchall())
else:
    print("Optional exercise skipped. Edit exercise_sql variable and set RUN_EXERCISE = True to run your query.")

ok("This cell complete!")


<details>
<summary>🔎 <b>Reveal solution: §1.A asset slice</b></summary>

```python
RUN_EXERCISE = True

exercise_sql = """
    SELECT a.name,
           a.asset_type,
           a.status,
           d.name AS district,
           a.criticality
    FROM infrastructure_assets a
    JOIN districts d ON d.district_id = a.district_id
    WHERE d.name = 'Harbor District'
    ORDER BY a.criticality DESC, a.name
"""

cursor.execute(exercise_sql)
cols = [c[0] for c in cursor.description]
show_table(cols, cursor.fetchall())
```

Try changing `Harbor District` to another district, or replace the `WHERE` clause with `WHERE a.asset_type = 'substation'`.

</details>


### JSON data (dot notation)

Each asset has a `specifications` column stored as the native JSON data type. Different asset types carry different technical attributes (span length for a bridge, voltage rating for a substation, diameter for a pipeline). JSON dot notation lets us extract these fields inline with standard SQL.

In [ ]:
# Extract JSON fields using dot notation alongside relational columns
cursor.execute("""
    SELECT a.name,
           a.asset_type,
           a.specifications.spanLength_m.number()   AS span_length_m,
           a.specifications.loadCapacity_t.number()  AS load_capacity_t,
           a.specifications.material.string()        AS material
    FROM infrastructure_assets a
    WHERE a.asset_type = 'bridge'
""")

cols = [c[0] for c in cursor.description]
show_table(cols, cursor.fetchall())

ok("This is an example of querying a JSON object stored in a JSON column in a relational table.")

> ✅ **Checkpoint: JSON and relational data work together**
>
> You queried JSON fields directly from SQL without moving the data into a separate document database.
>
> **Expected result:** asset rows with nested JSON attributes flattened into regular query results.


### Flattened specification helper view

The raw JSON column is useful when you need full flexibility. The helper view V_ASSET_SPECS_SUMMARY makes common fields easier to scan and compare while still preserving the original JSON document.


In [ ]:
# Compare common specification fields without writing every JSON path by hand
cursor.execute("""
    SELECT asset_name,
           asset_type,
           criticality,
           district_name,
           voltage_rating_kv,
           peak_capacity_mw,
           diameter_mm,
           span_length_m,
           backup_power_hours
    FROM v_asset_specs_summary
    WHERE asset_name IN (:bridge, :substation, :pipeline, :control_center, :eoc)
    ORDER BY criticality DESC, asset_name
""", {
    'bridge': DEMO_BRIDGE,
    'substation': DEMO_SUBSTATION,
    'pipeline': DEMO_PIPELINE,
    'control_center': DEMO_CONTROL_CENTER,
    'eoc': DEMO_EOC,
})

cols = [c[0] for c in cursor.description]
show_table(cols, cursor.fetchall())

ok("This is an example of how to compare common specification fields without")
ok("writing every JSON path by hand.")


In [ ]:
# ╭─ EXERCISE §1.B ─────────────────────────────────────────────────────╮
# │ TASK: Extract different JSON attributes for a non-bridge asset.     │
# │                                                                     │
# │ HOW TO COMPLETE THIS CELL:                                          │
# │   1. Query infrastructure_assets.                                   │
# │   2. Choose an asset_type such as 'substation', 'pipeline', or      │
# │      'sensor'.                                                      │
# │   3. Extract one or two fields from the specifications JSON column  │
# │      using dot notation.                                            │
# │   EXPECTED OUTPUT: asset rows plus JSON-derived columns.            │
# │   4. Set RUN_EXERCISE = True and run the cell.                      │
# │                                                                     │
# │ Stuck? Expand the 🔎 solution cell below and paste the full answer. │
# ╰─────────────────────────────────────────────────────────────────────╯

RUN_EXERCISE = False

exercise_sql = """
    -- TODO: query JSON fields from a.specifications for another asset type.
    -- Hint: use a.specifications.someField.string() or .number().
"""

if RUN_EXERCISE:
    cursor.execute(exercise_sql)
    cols = [c[0] for c in cursor.description]
    show_table(cols, cursor.fetchall())
    ok("Cell complete.")
else:
    print("Optional exercise skipped. Edit exercise_sql and set RUN_EXERCISE = True to run it.")


<details>
<summary>🔎 <b>Reveal solution: §1.B JSON dot notation</b></summary>

```python
RUN_EXERCISE = True

exercise_sql = """
    SELECT a.name,
           a.asset_type,
           a.specifications.voltageRating_kv.number() AS voltage_rating_kv,
           a.specifications.transformerCount.number() AS transformer_count,
           a.specifications.peakCapacity_mw.number() AS peak_capacity_mw,
           a.specifications.coolingType.string() AS cooling_type
    FROM infrastructure_assets a
    WHERE a.asset_type = 'substation'
    ORDER BY a.name
"""

cursor.execute(exercise_sql)
cols = [c[0] for c in cursor.description]
show_table(cols, cursor.fetchall())
ok("Cell complete.")
```

The substation documents use `voltageRating_kv`, `transformerCount`, `peakCapacity_mw`, and `coolingType`. If you choose another asset type, inspect its available specification keys before changing the JSON path.

</details>


### Property Graph (SQL/PGQ)

Asset connectivity (which pipeline feeds which substation, which sensor monitors which bridge) is stored in the `ASSET_CONNECTIONS` table and projected as the `CITYPULSE_GRAPH` property graph. SQL/PGQ `GRAPH_TABLE` syntax lets us query relationships directly.

In [ ]:
# Query the property graph for asset connections
cursor.execute("""
    SELECT *
    FROM GRAPH_TABLE (citypulse_graph
        MATCH (a IS asset) -[c IS connected_to]-> (b IS asset)
        COLUMNS (a.name           AS from_asset,
                 c.connection_type AS relationship,
                 b.name           AS to_asset)
    )
    FETCH FIRST 10 ROWS ONLY
""")

cols = [c[0] for c in cursor.description]
show_table(cols, cursor.fetchall())

ok("These are asset connections in the \"Property Graph\".")

More information about what [Property Graphs](https://en.wikipedia.org/wiki/Property_graph) are and their uses.

> ✅ **Checkpoint: Connected asset relationships are queryable**
>
> You used SQL/PGQ to traverse relationships between infrastructure assets.
>
> **Expected result:** connected assets and relationship labels that can also be visualized as a graph.


### Graph direction matters

Some relationships are directional. Substation Gamma has outgoing powers edges, while Harbor Bridge mainly receives incoming monitoring, support, and power relationships. Check direction before deciding whether to filter on from_asset or to_asset.


In [ ]:
# Compare outgoing and incoming edges for two demo assets
for target_asset in [DEMO_SUBSTATION, DEMO_BRIDGE]:
    cursor.execute("""
        WITH directed_edges AS (
            SELECT from_asset, relationship, to_asset
            FROM GRAPH_TABLE (citypulse_graph
                MATCH (a IS asset) -[c IS connected_to]-> (b IS asset)
                COLUMNS (a.name AS from_asset,
                         c.connection_type AS relationship,
                         b.name AS to_asset)
            )
        )
        SELECT 'outgoing' AS direction, from_asset, relationship, to_asset
        FROM directed_edges
        WHERE from_asset = :asset
        UNION ALL
        SELECT 'incoming' AS direction, from_asset, relationship, to_asset
        FROM directed_edges
        WHERE to_asset = :asset
        ORDER BY direction, relationship, from_asset, to_asset
    """, {'asset': target_asset})

    rows = cursor.fetchall()
    print(f"{target_asset}: {len(rows)} directed edge(s)")
    show_table(['Direction', 'From Asset', 'Relationship', 'To Asset'], rows)

ok("Notice the incoming and outgoing directions.")



In [ ]:
# ╭─ EXERCISE §1.C ─────────────────────────────────────────────────────╮
# │ TASK: traverse the graph from a named asset.                        │
# │                                                                     │
# │ HOW TO COMPLETE THIS CELL:                                          │
# │   1. Query CITYPULSE_GRAPH with GRAPH_TABLE.                        │
# │   2. Start from an asset with outgoing links, such as               │
# │      'Substation Gamma'.                                            │
# │   3. Return directly connected assets and relationship types.       │
# │   EXPECTED OUTPUT: from_asset, relationship, and to_asset.          │
# │   4. Set RUN_EXERCISE = True and run the cell.                      │
# │                                                                     │
# │ Stuck? Expand the 🔎 solution cell below and paste the full answer. │
# ╰─────────────────────────────────────────────────────────────────────╯

RUN_EXERCISE = False

exercise_sql = """
    -- TODO: use GRAPH_TABLE(citypulse_graph MATCH ... COLUMNS ...)
    -- Hint: filter the starting asset in the outer SELECT.
"""

if RUN_EXERCISE:
    cursor.execute(exercise_sql)
    cols = [c[0] for c in cursor.description]
    show_table(cols, cursor.fetchall())
    ok("Traversed the graph for an asset.")
else:
    ok("Optional exercise skipped. Edit exercise_sql and set RUN_EXERCISE = True to run it.")



<details>
<summary>🔎 <b>Reveal solution: §1.C graph traversal</b></summary>

```python
RUN_EXERCISE = True

exercise_sql = """
    SELECT from_asset, relationship, to_asset
    FROM GRAPH_TABLE (citypulse_graph
        MATCH (a IS asset) -[c IS connected_to]-> (b IS asset)
        COLUMNS (a.name AS from_asset,
                 c.connection_type AS relationship,
                 b.name AS to_asset)
    )
    WHERE from_asset = 'Substation Gamma'
    ORDER BY relationship, to_asset
"""

cursor.execute(exercise_sql)
cols = [c[0] for c in cursor.description]
show_table(cols, cursor.fetchall())
```

Try changing the `WHERE` clause to `to_asset = 'Harbor Bridge'` to inspect incoming relationships to Harbor Bridge.

</details>


In [ ]:
# Visualize the full asset connectivity graph
cursor.execute("""
    SELECT *
    FROM GRAPH_TABLE (citypulse_graph
        MATCH (a IS asset) -[c IS connected_to]-> (b IS asset)
        COLUMNS (a.name AS from_asset, c.connection_type AS rel, b.name AS to_asset)
    )
""")
graph_edges = [(r[0], r[1], r[2]) for r in cursor.fetchall()]

# Get asset types for node coloring
cursor.execute("SELECT name, asset_type FROM infrastructure_assets")
asset_types = {r[0]: r[1] for r in cursor.fetchall()}

show_graph(graph_edges, node_types=asset_types,
           title='Prism Infrastructure Connectivity Graph')

ok("A full visualization of the infrastructure and various connections.")

---
## Section 2: Vector embeddings with an in-database ONNX model

AI embedding models derive semantic meaning from text and output arrays of numbers called **vectors**. Semantically similar content produces vectors that are closer together in vector space.

### Embeddings stay inside the database

Oracle 26ai can load ONNX embedding models directly into the database. This means vector search runs entirely inside Oracle and **your data never leaves the database**. No external API calls, no data egress, no network latency for embedding generation. The model sits right next to your data.

### Verify the ONNX model

The ONNX embedding model configured in `ONNX_MODEL` has been pre-loaded into the database. Let's confirm it is available and that our user has access to it.

In [ ]:
# Verify the ONNX model is loaded and accessible
cursor.execute("""
    SELECT model_name, mining_function, algorithm, creation_date
    FROM all_mining_models
    WHERE model_name = :model_name
""", {"model_name": ONNX_MODEL.upper()})

result = cursor.fetchone()
if result:
    show_table(['Model Name', 'Mining Function', 'Algorithm', 'Created'], [result])
else:
    print(f"WARNING: Model {ONNX_MODEL} not found. Check that it has been loaded.")

ok("This cell complete!")


### Generate an embedding

Let's pass a piece of text through the model and see what comes back.

In [ ]:
# Generate a single vector embedding

print(f"This might take 20-30 seconds because of loading the model into memory first time.")
print(f"\nIf we were using GPU enabled instances and a more advanced model, this would be dramatically faster.")

cursor.execute(f"""
    SELECT VECTOR_EMBEDDING({ONNX_MODEL} USING
               'safety incident at pump station' AS data) AS embedding
    FROM DUAL
""")
embedding = cursor.fetchone()[0]

# Get the dimension count
cursor.execute(f"""
    SELECT VECTOR_DIMS(
        VECTOR_EMBEDDING({ONNX_MODEL} USING
            'safety incident at pump station' AS data)
    ) AS dimensions
    FROM DUAL
""")
dims = cursor.fetchone()[0]

print(f"\nTotal dimensions: {dims}")
print(f"\nFirst 200 characters of the vector:")
print(str(embedding)[:200], "...")

ok("This cell ran to completion!")

### Insert a new maintenance log and vectorize it

The Prism dataset already has vector embeddings pre-loaded in the `DOCUMENT_CHUNKS` table. Rather than re-vectorize everything (which would take a while), let's see the full pipeline in action by inserting a single new maintenance log and then creating the vector embeddings for it.

The pipeline has three steps:
1. **Insert** the maintenance log into the relational table
2. **Chunk** the narrative text using `DBMS_VECTOR_CHAIN.UTL_TO_CHUNKS`
3. **Embed** each chunk using `VECTOR_EMBEDDING` and store in `DOCUMENT_CHUNKS`

In [ ]:
# Step 1: Insert a new maintenance log for the demo bridge

# Look up the demo bridge asset_id
cursor.execute("SELECT asset_id FROM infrastructure_assets WHERE name = :name", {'name': DEMO_BRIDGE})
harbor_bridge_id = cursor.fetchone()[0]

narrative = (
    "Detected unusual vibration patterns on the north support cable during "
    "routine sensor sweep. Frequency analysis suggests possible fatigue stress "
    "at the cable anchor point near the western abutment. Corrosion visible on "
    "three secondary cable clamps. Recommending detailed structural inspection "
    "within 48 hours and temporary load restriction to single-lane traffic."
)

# Clean up any previous runs: remove chunks and logs with this narrative
cursor.execute("""
    DELETE FROM document_chunks
    WHERE source_table = 'maintenance_logs'
      AND source_id IN (
          SELECT log_id FROM maintenance_logs
          WHERE narrative LIKE '%north support cable%'
            AND asset_id = :asset_id
      )
""", {'asset_id': harbor_bridge_id})
chunks_cleaned = cursor.rowcount

cursor.execute("""
    DELETE FROM maintenance_logs
    WHERE narrative LIKE '%north support cable%'
      AND asset_id = :asset_id
""", {'asset_id': harbor_bridge_id})
logs_cleaned = cursor.rowcount

if logs_cleaned > 0:
    print(f"Cleaned up {logs_cleaned} previous log(s) and {chunks_cleaned} chunk(s) from prior runs.")

# Insert and retrieve the generated log_id
log_id_var = cursor.var(int)
cursor.execute("""
    INSERT INTO maintenance_logs (asset_id, log_date, severity, narrative)
    VALUES (:asset_id, SYSDATE, :severity, :narrative)
    RETURNING log_id INTO :log_id
""", {
    'asset_id': harbor_bridge_id,
    'severity': 'warning',
    'narrative': narrative,
    'log_id': log_id_var
})
new_log_id = log_id_var.getvalue()[0]
print(f"Step 1 complete: Inserted maintenance log with log_id = {new_log_id}")

# Step 2: Chunk the narrative
chunk_params = json.dumps({
    "max": 1000, "overlap": 100, "split": "sentence", "normalize": "all"
})

cursor.execute("""
    SELECT et.column_value
    FROM TABLE(
        DBMS_VECTOR_CHAIN.UTL_TO_CHUNKS(:input_text, JSON(:chunk_params))
    ) et
""", {'input_text': narrative, 'chunk_params': chunk_params})
chunks_raw = cursor.fetchall()
print(f"Step 2 complete: Created {len(chunks_raw)} chunk(s)")

# Step 3: Embed each chunk and insert into DOCUMENT_CHUNKS
for idx, (chunk_value,) in enumerate(chunks_raw, start=1):
    # UTL_TO_CHUNKS returns JSON objects; extract the chunk_data field
    if isinstance(chunk_value, str):
        try:
            chunk_json = json.loads(chunk_value)
            chunk_text = chunk_json.get("chunk_data", chunk_value)
        except json.JSONDecodeError:
            chunk_text = chunk_value
    else:
        chunk_text = str(chunk_value)

    cursor.execute(f"""
        INSERT INTO document_chunks (
            source_table, source_id, source_key, source_label, source_date,
            chunk_seq, chunk_text, model_name, chunk_params, embedding
        ) VALUES (
            :source_table, :source_id, :source_key, :source_label, SYSDATE,
            :chunk_seq, :chunk_text, :model_name, JSON(:chunk_params),
            VECTOR_EMBEDDING({ONNX_MODEL} USING :chunk_text AS data)
        )
    """, {
        'source_table': 'maintenance_logs',
        'source_id': new_log_id,
        'source_key': str(new_log_id),
        'source_label': DEMO_BRIDGE,
        'chunk_seq': idx,
        'chunk_text': chunk_text,
        'model_name': ONNX_MODEL.upper(),
        'chunk_params': chunk_params
    })

conn.commit()
print(f"Step 3 complete: Embedded and stored {len(chunks_raw)} chunk(s) in DOCUMENT_CHUNKS table.")

ok("All steps in this cell completed.")

### Verify the new vectors

Let's query `DOCUMENT_CHUNKS` for the log we just inserted and confirm the chunks and embeddings exist.

In [ ]:
# Verify the chunks and embeddings were created
cursor.execute("""
    SELECT dc.chunk_id,
           dc.chunk_seq,
           SUBSTR(dc.chunk_text, 1, 100) AS chunk_preview,
           VECTOR_DIMS(dc.embedding)     AS dimensions
    FROM document_chunks dc
    WHERE dc.source_table = 'maintenance_logs'
      AND dc.source_id = :log_id
    ORDER BY dc.chunk_seq
""", {'log_id': new_log_id})

cols = [c[0] for c in cursor.description]
show_table(cols, cursor.fetchall())

ok("Cell complete!")

---
## Section 3: Inspect the vector index

To perform fast [approximate nearest neighbor (ANN) search](https://en.wikipedia.org/wiki/Nearest_neighbor_search) over the embeddings, Prism uses an **HNSW** (Hierarchical Navigable Small Worlds) vector index. HNSW builds a multi-layered in-memory graph structure that enables high-recall similarity search.

The readiness probe creates `IDX_CHUNK_EMBEDDING` automatically if it is missing. This section inspects the index and shows the DDL pattern for reference without dropping or rebuilding it.

We use **cosine distance**, which measures the angle between vectors. It focuses on meaning rather than magnitude, so a short sentence and a long paragraph about the same topic can still be close.


In [ ]:
# Inspect the vector index and keep the DDL as a reference.
index_ddl_example = """
CREATE VECTOR INDEX idx_chunk_embedding
    ON document_chunks(embedding)
    ORGANIZATION INMEMORY NEIGHBOR GRAPH
    DISTANCE COSINE
    WITH TARGET ACCURACY 95
""".strip()

cursor.execute("""
    SELECT index_name, index_type, status
    FROM user_indexes
    WHERE index_name = 'IDX_CHUNK_EMBEDDING'
""")
index_rows = cursor.fetchall()

if index_rows:
    show_table([c[0] for c in cursor.description], index_rows)
    ok("Vector index IDX_CHUNK_EMBEDDING is present.")
else:
    cursor.execute(index_ddl_example)
    cursor.execute("""
        SELECT index_name, index_type, status
        FROM user_indexes
        WHERE index_name = 'IDX_CHUNK_EMBEDDING'
    """)
    index_rows = cursor.fetchall()
    show_table([c[0] for c in cursor.description], index_rows)
    ok("Vector index IDX_CHUNK_EMBEDDING was created.")

print("\nReference DDL for this index:")
print(index_ddl_example)


In [ ]:
# Verify index column and distance configuration metadata where available.
cursor.execute("""
    SELECT ic.index_name,
           ic.table_name,
           ic.column_name,
           ui.status
    FROM user_ind_columns ic
    JOIN user_indexes ui ON ui.index_name = ic.index_name
    WHERE ic.index_name = 'IDX_CHUNK_EMBEDDING'
    ORDER BY ic.column_position
""")

cols = [c[0] for c in cursor.description]
rows = cursor.fetchall()
if rows:
    show_table(cols, rows)
    ok("Index verified.")
else:
    ok("No index metadata found for IDX_CHUNK_EMBEDDING.")


In [ ]:
# Prove the HNSW vector index is used by the indexed top-k query shape.
plan_probe_text = "structural damage and corrosion on bridge assets"

cursor.execute(f"""
    EXPLAIN PLAN FOR
    SELECT /*+ VECTOR_INDEX_TRANSFORM(dc idx_chunk_embedding IN_FILTER_WITH_JOIN_BACK) */
           dc.chunk_id,
           VECTOR_DISTANCE(dc.embedding,
               VECTOR_EMBEDDING({ONNX_MODEL} USING 'structural damage and corrosion on bridge assets' AS data),
               COSINE) AS distance
    FROM document_chunks dc
    -- Problem/evidence searches should rank observations, not SOPs.
    -- SOP chunks answer what to do next, so we search them separately.
    WHERE dc.source_table IN (
        'maintenance_logs',
        'inspection_reports',
        'inspection_findings'
    )
    ORDER BY VECTOR_DISTANCE(dc.embedding,
        VECTOR_EMBEDDING({ONNX_MODEL} USING 'structural damage and corrosion on bridge assets' AS data),
        COSINE)
    FETCH APPROX FIRST 5 ROWS ONLY WITH TARGET ACCURACY 95
""")

cursor.execute("""
    SELECT plan_table_output
    FROM TABLE(DBMS_XPLAN.DISPLAY(NULL, NULL, 'BASIC +PREDICATE'))
""")
plan_text = "\n".join(row[0] for row in cursor.fetchall())
print(plan_text)

if "VECTOR INDEX HNSW SCAN" not in plan_text.upper():
    raise RuntimeError("Expected VECTOR INDEX HNSW SCAN in the execution plan.")

ok("This cell complete!")


> ✅ **Checkpoint: Vector search is ready**
>
> The embedding model is available, maintenance text has been vectorized, the vector index is present, and the execution plan confirms `VECTOR INDEX HNSW SCAN` for the indexed top-k query shape.
>
> **Expected result:** the next semantic search returns records ranked by meaning, then joins the top vector hits back to the unified context view.


---
## Section 4: Vector search

### A beginning vector search

In [ ]:
# ╭─ EXERCISE §3.A ─────────────────────────────────────────────────────╮
# │ TASK: ask a different semantic search question.                     │
# │                                                                     │
# │ HOW TO COMPLETE THIS CELL:                                          │
# │   1. Replace QUESTION with your own natural-language prompt.        │
# │   2. Keep the indexed DOCUMENT_CHUNKS top-k pattern.                │
# │   EXPECTED OUTPUT: asset_name, source_table, chunk_text, distance.   │
# │   3. Set RUN_EXERCISE = True and run the cell.                      │
# │                                                                     │
# │ Stuck? Expand the 🔎 solution cell below and paste the full answer. │
# ╰─────────────────────────────────────────────────────────────────────╯

RUN_EXERCISE = False
QUESTION = "TODO: write a semantic search question here"

if RUN_EXERCISE:
    cursor.execute(f"""
        WITH top_chunks AS (
            SELECT /*+ VECTOR_INDEX_TRANSFORM(dc idx_chunk_embedding IN_FILTER_WITH_JOIN_BACK) */
                   dc.chunk_id,
                   VECTOR_DISTANCE(dc.embedding,
                       VECTOR_EMBEDDING({ONNX_MODEL} USING :question AS data),
                       COSINE) AS distance
            FROM document_chunks dc
            ORDER BY VECTOR_DISTANCE(dc.embedding,
                VECTOR_EMBEDDING({ONNX_MODEL} USING :question AS data),
                COSINE)
            FETCH APPROX FIRST 5 ROWS ONLY WITH TARGET ACCURACY 95
        )
        SELECT u.asset_name,
               u.source_table,
               u.chunk_text,
               ROUND(t.distance, 4) AS distance
        FROM top_chunks t
        JOIN v_chunks_unified u
          ON u.chunk_id = t.chunk_id
        ORDER BY t.distance
    """, {"question": QUESTION})
    cols = [c[0] for c in cursor.description]
    show_table(cols, cursor.fetchall(), max_width=100)
    ok("Vector search complete.")
else:
    ok("Optional exercise skipped. Set QUESTION, then set RUN_EXERCISE = True to run it.")


<details>
<summary>🔎 <b>Reveal solution: §3.A semantic search sample question</b></summary>

Try a question that does **not** use exact table terms, such as “Which assets look vulnerable during severe weather?” Semantic search should still find related content.

</details>


---
### Targeted vector search

Now let's try a more targeted search. We'll embed a natural language question and find the most semantically similar evidence across maintenance logs, inspection reports, and inspection findings for a specific asset.

Operational procedures have embeddings in the `DOCUMENT_CHUNKS` table too, but they answer a different question: "what should we do next?" We search those standard operating procedures (SOPs) chunks in a separate procedure-focused example below. Therefore we filter them out in the `WHERE` clause.


In [ ]:
# Semantic search: structural damage on the demo bridge
search_text = f"structural damage and corrosion on {DEMO_BRIDGE}"

cursor.execute(f"""
    WITH top_chunks AS (
        SELECT /*+ VECTOR_INDEX_TRANSFORM(dc idx_chunk_embedding IN_FILTER_WITH_JOIN_BACK) */
               dc.chunk_id,
               VECTOR_DISTANCE(dc.embedding,
                   VECTOR_EMBEDDING({ONNX_MODEL} USING :search_text AS data),
                   COSINE) AS distance
        FROM document_chunks dc
        -- Problem/evidence searches should rank observations, not SOPs.
        -- SOP chunks answer what to do next, so we search them separately.
        WHERE dc.source_table IN (
            'maintenance_logs',
            'inspection_reports',
            'inspection_findings'
        )
        ORDER BY VECTOR_DISTANCE(dc.embedding,
            VECTOR_EMBEDDING({ONNX_MODEL} USING :search_text AS data),
            COSINE)
        FETCH APPROX FIRST 5 ROWS ONLY WITH TARGET ACCURACY 95
    )
    SELECT u.source_table,
           u.source_date,
           SUBSTR(u.chunk_text, 1, 240) AS chunk_preview,
           u.asset_name,
           u.district_name,
           u.criticality,
           ROUND(t.distance, 4) AS distance
    FROM top_chunks t
    JOIN v_chunks_unified u
      ON u.chunk_id = t.chunk_id
    ORDER BY t.distance, u.criticality DESC NULLS LAST
""", {'search_text': search_text})

cols = [c[0] for c in cursor.description]
show_table(cols, cursor.fetchall())

ok("This cell complete!")


**How to read the results:** Cosine distance closer to 0 means more semantically similar. Notice that the results are relevant even when the exact search words don't appear in the source text. That is the power of semantic search: it matches on *meaning*, not keywords.

Also notice the `SOURCE_TABLE` column. These evidence searches intentionally include maintenance logs, inspection reports, and inspection findings, while excluding operational procedures so SOP guidance does not crowd out observed infrastructure problems.


### Procedure embeddings: finding the right SOP

The data model embeds JSON documents in `OPERATIONAL_PROCEDURES`. The procedure-specific chunk view keeps this search focused on the standard operating procedure content, while the view `V_OPERATIONAL_PROCEDURE_SUMMARY` projects key JSON fields into SQL columns when those are needed. Same data, different shapes as consumers need them.


In [ ]:
# Semantic search over operational procedure embeddings
sop_question = "There's water everywhere!! What's the procedure for a pipeline leak with emergency response steps?"

cursor.execute(f"""
    WITH top_chunks AS (
        SELECT /*+ VECTOR_INDEX_TRANSFORM(dc idx_chunk_embedding IN_FILTER_WITH_JOIN_BACK) */
               dc.chunk_id,
               dc.source_key AS procedure_id,
               VECTOR_DISTANCE(dc.embedding,
                   VECTOR_EMBEDDING({ONNX_MODEL} USING :question AS data),
                   COSINE) AS distance
        FROM document_chunks dc
        -- This query intentionally searches just SOP chunks because we know
        -- the question is about which procedure applies, not what evidence exists.
        WHERE dc.source_table = 'operational_procedures'
        ORDER BY VECTOR_DISTANCE(dc.embedding,
            VECTOR_EMBEDDING({ONNX_MODEL} USING :question AS data),
            COSINE)
        FETCH APPROX FIRST 5 ROWS ONLY WITH TARGET ACCURACY 95
    )
    SELECT p.procedure_id,
           p.title,
           p.category,
           p.escalation_contact,
           SUBSTR(c.chunk_text, 1, 240) AS chunk_preview,
           ROUND(t.distance, 4) AS distance
    FROM top_chunks t
    JOIN document_chunks c
      ON c.chunk_id = t.chunk_id
    JOIN v_operational_procedure_summary p
      ON p.procedure_id = t.procedure_id
    ORDER BY t.distance
""", {"question": sop_question})

print(f"SOP question: {sop_question}")
cols = [c[0] for c in cursor.description]
show_table(cols, cursor.fetchall(), max_width=100)

ok("Query complete.")


This next query uses the same database content viewed through two surfaces: the SOP remains a JSON document, and its vectorized embedded chunks make it searchable by operational meaning. That is the converged-database pattern: one logical model, multiple access paths.


In [ ]:
# Semantic search: use a known-good scenario from the guide
scenario = SCENARIOS[2]

cursor.execute(f"""
    WITH top_chunks AS (
        SELECT /*+ VECTOR_INDEX_TRANSFORM(dc idx_chunk_embedding IN_FILTER_WITH_JOIN_BACK) */
               dc.chunk_id,
               VECTOR_DISTANCE(dc.embedding,
                   VECTOR_EMBEDDING({ONNX_MODEL} USING :question AS data),
                   COSINE) AS distance
        FROM document_chunks dc
        -- Problem/evidence searches should rank observations, not SOPs.
        -- SOP chunks answer what to do next, so we search them separately.
        WHERE dc.source_table IN (
            'maintenance_logs',
            'inspection_reports',
            'inspection_findings'
        )
        ORDER BY VECTOR_DISTANCE(dc.embedding,
            VECTOR_EMBEDDING({ONNX_MODEL} USING :question AS data),
            COSINE)
        FETCH APPROX FIRST 5 ROWS ONLY WITH TARGET ACCURACY 95
    )
    SELECT u.source_table,
           u.source_date,
           SUBSTR(u.chunk_text, 1, 240) AS chunk_preview,
           u.asset_name,
           u.district_name,
           u.criticality,
           ROUND(t.distance, 4) AS distance
    FROM top_chunks t
    JOIN v_chunks_unified u
      ON u.chunk_id = t.chunk_id
    ORDER BY t.distance, u.criticality DESC NULLS LAST
""", {'question': scenario['question']})

print(f"Scenario: {scenario['name']}")
cols = [c[0] for c in cursor.description]
show_table(cols, cursor.fetchall())

ok("This cell complete!")


---
## Section 5: The unified query: Four data shapes, one SQL statement

In a polyglot persistence world, the query you're about to run would require four separate databases, four network hops, and a synchronization strategy. In Oracle 26ai, it's **one query, one transaction, zero syncing**.

This query will:
1. **Vector search** finds semantically similar maintenance and inspection content
2. **Relational JOINs** pull in asset details, district info, and inspection history
3. **JSON dot notation** extracts technical specifications from the JSON column
4. **Graph traversal** finds physically connected assets

### Query 1: Harbor bridge (Table output)

In [ ]:
# Unified query: relational + JSON + graph + vector in one SQL statement
bridge_question = f"maintenance issues and structural condition of {DEMO_BRIDGE}"

cursor.execute(f"""
    WITH top_chunks AS (
        SELECT /*+ VECTOR_INDEX_TRANSFORM(dc idx_chunk_embedding IN_FILTER_WITH_JOIN_BACK) */
               dc.chunk_id,
               VECTOR_DISTANCE(dc.embedding,
                   VECTOR_EMBEDDING({ONNX_MODEL} USING :question AS data),
                   COSINE) AS distance
        FROM document_chunks dc
        -- Problem/evidence searches should rank observations, not SOPs.
        -- SOP chunks answer what to do next, so we search them separately.
        WHERE dc.source_table IN (
            'maintenance_logs',
            'inspection_reports',
            'inspection_findings'
        )
        ORDER BY VECTOR_DISTANCE(dc.embedding,
            VECTOR_EMBEDDING({ONNX_MODEL} USING :question AS data),
            COSINE)
        FETCH APPROX FIRST 5 ROWS ONLY WITH TARGET ACCURACY 95
    ),
    vector_hits AS (
        SELECT u.source_table, u.source_id, u.chunk_text,
               u.asset_id, u.asset_name, u.district_name, u.criticality,
               t.distance
        FROM top_chunks t
        JOIN v_chunks_unified u
          ON u.chunk_id = t.chunk_id
    ),
    directed_edges AS (
        SELECT from_asset, relationship, to_asset
        FROM GRAPH_TABLE (citypulse_graph
            MATCH (a IS asset) -[c IS connected_to]-> (b IS asset)
            COLUMNS (a.name AS from_asset,
                     c.connection_type AS relationship,
                     b.name AS to_asset)
        )
    ),
    connected AS (
        SELECT to_asset AS connected_asset, relationship
        FROM directed_edges
        WHERE from_asset = :asset
        UNION ALL
        SELECT from_asset AS connected_asset, relationship
        FROM directed_edges
        WHERE to_asset = :asset
    )
    SELECT vh.source_table                           AS source,
           SUBSTR(vh.chunk_text, 1, 100)             AS content_preview,
           ROUND(vh.distance, 4)                     AS vector_dist,
           ia.name                                   AS asset,
           d.name                                    AS district,
           ia.criticality                            AS criticality,
           ia.specifications.spanLength_m.number()   AS span_m,
           ia.specifications.loadCapacity_t.number() AS capacity_t,
           ia.specifications.material.string()       AS material,
           (SELECT LISTAGG(c.connected_asset || ' (' || c.relationship || ')', ', ')
                   WITHIN GROUP (ORDER BY c.connected_asset)
            FROM connected c)                        AS connected_assets
    FROM vector_hits vh
    JOIN infrastructure_assets ia ON ia.asset_id = vh.asset_id
    JOIN districts d ON d.district_id = ia.district_id
    ORDER BY vh.distance, ia.criticality DESC
""", {'asset': DEMO_BRIDGE, 'question': bridge_question})

cols = [c[0] for c in cursor.description]
show_table(cols, cursor.fetchall())

ok("This cell completed.")


### Query 1: Same results, but output in JSON format

Oracle can also project those same results as a JSON document. Same query, different output shape.

In [ ]:
# The same unified query, but output as a JSON document
bridge_question = f"maintenance issues and structural condition of {DEMO_BRIDGE}"

cursor.execute(f"""
    WITH top_chunks AS (
        SELECT /*+ VECTOR_INDEX_TRANSFORM(dc idx_chunk_embedding IN_FILTER_WITH_JOIN_BACK) */
               dc.chunk_id,
               VECTOR_DISTANCE(dc.embedding,
                   VECTOR_EMBEDDING({ONNX_MODEL} USING :question AS data),
                   COSINE) AS distance
        FROM document_chunks dc
        -- Problem/evidence searches should rank observations, not SOPs.
        -- SOP chunks answer what to do next, so we search them separately.
        WHERE dc.source_table IN (
            'maintenance_logs',
            'inspection_reports',
            'inspection_findings'
        )
        ORDER BY VECTOR_DISTANCE(dc.embedding,
            VECTOR_EMBEDDING({ONNX_MODEL} USING :question AS data),
            COSINE)
        FETCH APPROX FIRST 5 ROWS ONLY WITH TARGET ACCURACY 95
    ),
    vector_hits AS (
        SELECT u.source_table, u.source_id, u.chunk_text, u.asset_id,
               u.asset_name, u.district_name, u.criticality,
               t.distance
        FROM top_chunks t
        JOIN v_chunks_unified u
          ON u.chunk_id = t.chunk_id
    ),
    directed_edges AS (
        SELECT from_asset, relationship, to_asset
        FROM GRAPH_TABLE (citypulse_graph
            MATCH (a IS asset) -[c IS connected_to]-> (b IS asset)
            COLUMNS (a.name AS from_asset,
                     c.connection_type AS relationship,
                     b.name AS to_asset)
        )
    ),
    connected AS (
        SELECT to_asset AS connected_asset, relationship
        FROM directed_edges
        WHERE from_asset = :asset
        UNION ALL
        SELECT from_asset AS connected_asset, relationship
        FROM directed_edges
        WHERE to_asset = :asset
    ),
    result_data AS (
        SELECT vh.source_table, vh.chunk_text, vh.distance,
               ia.name AS asset_name, ia.criticality, d.name AS district,
               ia.specifications AS specs
        FROM vector_hits vh
        JOIN infrastructure_assets ia ON ia.asset_id = vh.asset_id
        JOIN districts d ON d.district_id = ia.district_id
    )
    SELECT JSON_OBJECT(
            'query' VALUE :question,
            'results' VALUE (
                SELECT JSON_ARRAYAGG(
                    JSON_OBJECT(
                        'source'      VALUE r.source_table,
                        'preview'     VALUE SUBSTR(r.chunk_text, 1, 150),
                        'distance'    VALUE ROUND(r.distance, 4),
                        'asset'       VALUE r.asset_name,
                        'criticality' VALUE r.criticality,
                        'district'    VALUE r.district,
                        'specs'       VALUE r.specs
                    ) ORDER BY r.distance
                ) FROM result_data r
            ),
            'connected_assets' VALUE (
                SELECT JSON_ARRAYAGG(
                    JSON_OBJECT(
                        'asset'           VALUE c.connected_asset,
                        'connection_type' VALUE c.relationship
                    )
                ) FROM connected c
            )
        RETURNING JSON
    ) AS json_output
    FROM DUAL
""", {'asset': DEMO_BRIDGE, 'question': bridge_question})

result = cursor.fetchone()[0]
print_json(result)

ok("This cell complete!")


> ✅ **Checkpoint: One query, multiple data shapes**
>
> You combined relational filters, JSON attributes, graph relationships, and vector similarity in one SQL statement.
>
> **Expected result:** a compact incident-style result for `Harbor Bridge`, including related assets and relevant maintenance context.


In [ ]:
# ╭─ EXERCISE §5.A ─────────────────────────────────────────────────────╮
# │ TASK: rerun the unified query for a different asset.                │
# │                                                                     │
# │ HOW TO COMPLETE THIS CELL:                                          │
# │   1. Pick a target asset.                                           │
# │   2. Ask a question about risk, root cause, or next actions.        │
# │   3. Return relational, JSON, graph, and vector context together.   │
# │   EXPECTED OUTPUT: one JSON document with asset and context fields.  │
# │   4. Set RUN_EXERCISE = True and run the cell.                      │
# │                                                                     │
# │ Stuck? Expand the 🔎 solution cell below and paste the full answer. │
# ╰─────────────────────────────────────────────────────────────────────╯

RUN_EXERCISE = False
TARGET_ASSET = "TODO: asset name"
QUESTION = "TODO: natural-language question"

if RUN_EXERCISE:
    cursor.execute(f"""
        WITH graph_neighbors AS (
            SELECT from_asset, relationship, to_asset
            FROM GRAPH_TABLE (citypulse_graph
                MATCH (a IS asset) -[c IS connected_to]-> (b IS asset)
                COLUMNS (a.name AS from_asset,
                         c.connection_type AS relationship,
                         b.name AS to_asset)
            )
            WHERE from_asset = :asset
        ),
        top_chunks AS (
            SELECT /*+ VECTOR_INDEX_TRANSFORM(dc idx_chunk_embedding IN_FILTER_WITH_JOIN_BACK) */
                   dc.chunk_id,
                   VECTOR_DISTANCE(dc.embedding,
                       VECTOR_EMBEDDING({ONNX_MODEL} USING :question AS data),
                       COSINE) AS distance
            FROM document_chunks dc
            -- Keep this asset-risk search focused on evidence rows.
            -- SOP rows are searched in their own procedure-specific retrieval step.
            WHERE dc.source_table IN (
                'maintenance_logs',
                'inspection_reports',
                'inspection_findings'
            )
              AND EXISTS (
                SELECT 1
                FROM v_chunks_unified u
                WHERE u.chunk_id = dc.chunk_id
                  AND u.asset_name = :asset
            )
            ORDER BY VECTOR_DISTANCE(dc.embedding,
                VECTOR_EMBEDDING({ONNX_MODEL} USING :question AS data),
                COSINE)
            FETCH APPROX FIRST 3 ROWS ONLY WITH TARGET ACCURACY 95
        ),
        semantic_context AS (
            SELECT u.asset_name, u.source_table, u.chunk_text, t.distance
            FROM top_chunks t
            JOIN v_chunks_unified u
              ON u.chunk_id = t.chunk_id
        )
        SELECT JSON_SERIALIZE(
            JSON_OBJECT(
                'asset' VALUE a.name,
                'type' VALUE a.asset_type,
                'status' VALUE a.status,
                'district' VALUE d.name,
                'specifications' VALUE a.specifications,
                'connected_assets' VALUE (
                    SELECT JSON_ARRAYAGG(JSON_OBJECT(
                        'relationship' VALUE relationship,
                        'to_asset' VALUE to_asset
                    )) FROM graph_neighbors
                ),
                'semantic_context' VALUE (
                    SELECT JSON_ARRAYAGG(JSON_OBJECT(
                        'source' VALUE source_table,
                        'text' VALUE chunk_text,
                        'distance' VALUE distance
                    )) FROM semantic_context
                )
            ) PRETTY
        ) AS asset_context
        FROM infrastructure_assets a
        JOIN districts d ON d.district_id = a.district_id
        WHERE a.name = :asset
    """, {"asset": TARGET_ASSET, "question": QUESTION})
    print_json(cursor.fetchone()[0])
    ok("Done!")
else:
    ok("Optional exercise skipped. Set TARGET_ASSET and QUESTION, then set RUN_EXERCISE = True to run it.")


<details>
<summary>🔎 <b>Reveal solution: §5.A unified query remix</b></summary>

```python
RUN_EXERCISE = True
TARGET_ASSET = "Substation Gamma"
QUESTION = "What recent evidence points to electrical instability and what connected assets may be affected?"
```

This is the main Prism pattern: one SQL statement can combine structured attributes, flexible JSON specs, graph relationships, and semantic context.

</details>


**One database. One query. Four data shapes. Zero synchronization tax.**

### Query 2: Graph-driven discovery

This query flips the perspective. Instead of starting from a known asset, we start from critical incidents and use graph traversal to discover which *other* assets are connected to the ones that had problems. Then we pull their inspection records and specs to assess downstream risk.

In [ ]:
# Graph-driven discovery: start from critical incidents, traverse outward
cursor.execute(f"""
    WITH critical_assets AS (
        SELECT DISTINCT ia.name AS asset_name
        FROM maintenance_logs ml
        JOIN infrastructure_assets ia ON ia.asset_id = ml.asset_id
        WHERE ml.severity = 'critical'
    ),
    all_connections AS (
        SELECT *
        FROM GRAPH_TABLE (citypulse_graph
            MATCH (a IS asset) -[c IS connected_to]- (b IS asset)
            COLUMNS (
                a.name           AS from_asset,
                c.connection_type AS connection_type,
                b.name           AS to_asset,
                b.asset_type     AS to_type
            )
        )
    ),
    impacted AS (
        SELECT DISTINCT
               ac.from_asset, ac.connection_type,
               ac.to_asset, ac.to_type
        FROM all_connections ac
        JOIN critical_assets ca ON ca.asset_name = ac.from_asset
    ),
    latest_inspections AS (
        SELECT ir.asset_id, ir.overall_grade, ir.summary,
               ROW_NUMBER() OVER (
                   PARTITION BY ir.asset_id ORDER BY ir.inspect_date DESC
               ) AS rn
        FROM inspection_reports ir
    )
    SELECT imp.from_asset                   AS critical_asset,
           imp.connection_type               AS relationship,
           imp.to_asset                      AS connected_asset,
           imp.to_type                       AS asset_type,
           spec.criticality                  AS criticality,
           spec.voltage_rating_kv            AS voltage_kv,
           spec.diameter_mm                  AS diameter_mm,
           spec.height_m                     AS height_m,
           li.overall_grade                  AS last_grade,
           SUBSTR(li.summary, 1, 100)        AS inspection_summary
    FROM impacted imp
    JOIN infrastructure_assets ia ON ia.name = imp.to_asset
    JOIN v_asset_specs_summary spec ON spec.asset_id = ia.asset_id
    LEFT JOIN latest_inspections li
        ON li.asset_id = ia.asset_id AND li.rn = 1
    ORDER BY spec.criticality DESC, imp.from_asset, imp.connection_type
""")

cols = [c[0] for c in cursor.description]
rows = cursor.fetchall()
if rows:
    show_table(cols, rows)
    ok("Query complete.")
else:
    ok("No critical-severity maintenance logs found.")
    print("Tip: Change the severity filter to 'warning' and re-run.")



In [ ]:
# Visualize the critical assets and their connected neighbors
if rows:
    viz_edges = [(r[0], r[1], r[2]) for r in rows]

    # Build node type map from the results
    viz_types = {}
    critical_names = set()
    for r in rows:
        critical_names.add(r[0])
        viz_types[r[2]] = r[3]  # connected_asset -> asset_type
    # Look up types for the critical assets too
    for name in critical_names:
        cursor.execute("SELECT asset_type FROM infrastructure_assets WHERE name = :n", {'n': name})
        result = cursor.fetchone()
        if result:
            viz_types[name] = result[0]

    show_graph(viz_edges, node_types=viz_types,
              highlight_nodes=critical_names,
              title='Critical Incident Impact: Connected Assets')
    ok("\nQuery complete!")
else:
    ok('No graph to display (no critical-severity logs found).')


---
## Section 6: OPTIONAL: Hybrid Vector Search (+15 minutes)

**This section is optional.** If you have time, it demonstrates how Oracle's Hybrid Vector Index combines lexical keyword search with semantic vector search in a single index.

### The problem

Vector search excels at semantic similarity but can miss exact matches on asset identifiers, codes, or specific terminology. Lexical search nails exact matches but misses semantically related content that uses different wording.

Consider: *"What safety incidents involved Substation Gamma and what were the root causes?"*

This needs **semantic understanding** of "root causes" AND **exact matching** on "Substation Gamma." Neither search alone gets it right. Hybrid search combines both and fuses the scores.

> 🧭 **Optional section: Hybrid Vector Search**
>
> **Time estimate:** 10-15 minutes.
>
> This section compares vector-only search, text-only search, and hybrid search.
>
> **Skip safely:** You can skip to cleanup and still complete the main workshop story. The first hybrid query can be slower while Oracle initializes the search structures.


### Create a dedicated hybrid search table

We create a separate table for this demo so the hybrid index has a clean text column to work with.

In [ ]:
# Optional hybrid section gate. Leave False for normal run-all execution.
RUN_HYBRID_SECTION = True

def safe_execute_ddl(sql, label):
    try:
        cursor.execute(sql)
        print(f"{label}: done")
    except Exception as exc:
        print(f"{label}: skipped ({type(exc).__name__}: {str(exc)[:100]})")

if RUN_HYBRID_SECTION:
    # Drop hybrid search objects if they exist (for re-runs)
    safe_execute_ddl("DROP INDEX idx_hybrid_demo FORCE", "Drop IDX_HYBRID_DEMO")
    safe_execute_ddl("DROP TABLE hybrid_search_demo PURGE", "Drop HYBRID_SEARCH_DEMO")
    safe_execute_ddl("BEGIN DBMS_VECTOR_CHAIN.DROP_PREFERENCE('prism_hybrid_pref'); END;", "Drop PRISM_HYBRID_PREF")

    # Create and populate the hybrid search demo table
    cursor.execute("""
        CREATE TABLE hybrid_search_demo (
            doc_id      NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
            asset_name  VARCHAR2(200),
            severity    VARCHAR2(20),
            source_type VARCHAR2(50),
            content     VARCHAR2(4000) NOT NULL
        )
    """)

    cursor.execute("""
        INSERT INTO hybrid_search_demo (asset_name, severity, source_type, content)
        SELECT ia.name, ml.severity, 'maintenance_log', ml.narrative
        FROM maintenance_logs ml
        JOIN infrastructure_assets ia ON ia.asset_id = ml.asset_id
    """)
    log_count = cursor.rowcount

    cursor.execute("""
        INSERT INTO hybrid_search_demo (asset_name, severity, source_type, content)
        SELECT ia.name, inf.severity, 'inspection_finding', inf.description
        FROM inspection_findings inf
        JOIN inspection_reports ir ON ir.report_id = inf.report_id
        JOIN infrastructure_assets ia ON ia.asset_id = ir.asset_id
        WHERE inf.description IS NOT NULL
    """)
    finding_count = cursor.rowcount

    conn.commit()
    print(f"Table created and populated: {log_count} maintenance logs + {finding_count} inspection findings")
    ok("This cell complete!")
else:
    ok("Optional hybrid section skipped. Set RUN_HYBRID_SECTION = True in this cell to run it.")


### Create the vectorizer preference and hybrid index

A **vectorizer preference** tells Oracle how to chunk and embed text inside the hybrid index. The hybrid index then builds both a full-text (Oracle Text) index and a vector index in one unified structure.

In [ ]:
if RUN_HYBRID_SECTION:
    # Create the vectorizer preference
    cursor.execute(f"""
        BEGIN
            DBMS_VECTOR_CHAIN.CREATE_PREFERENCE(
                'prism_hybrid_pref',
                DBMS_VECTOR_CHAIN.VECTORIZER,
                JSON('{{"vector_idxtype": "hnsw",
                       "model":          "{ONNX_MODEL}",
                       "by":             "words",
                       "max":            100,
                       "overlap":         10,
                       "split":          "recursively"}}')
            );
        END;
    """)
    print("Vectorizer preference created.")

    # Create the hybrid vector index (this may take a moment)
    print("Creating hybrid vector index (this may take a moment)...")
    cursor.execute("""
        CREATE HYBRID VECTOR INDEX idx_hybrid_demo
            ON hybrid_search_demo(content)
            PARAMETERS('VECTORIZER prism_hybrid_pref')
    """)
    print("Hybrid vector index idx_hybrid_demo created.")
    print()
    print("Note: The hybrid index uses MAINTENANCE AUTO, which means Oracle is")
    print("chunking, embedding, and indexing all rows in the background.")
    print("The first search query may take 30-60 seconds while this completes.")

    ok("Hybrid index creation is complete")
else:
    ok("Optional hybrid section skipped: set RUN_HYBRID_SECTION = True in the first hybrid cell to run hybrid index creation.")


### Vector-only search

First, let's see what a pure semantic search returns through the hybrid index.

> **Heads up:** The first query after index creation may take 30-60 seconds. The hybrid index is finishing its background synchronization (chunking, embedding, and indexing all rows). Subsequent queries will be fast.

In [ ]:
if RUN_HYBRID_SECTION:
    # Pure vector-only search via the hybrid index
    cursor.execute("""
        SELECT DBMS_HYBRID_VECTOR.SEARCH(
                JSON('{
                    "hybrid_index_name" : "IDX_HYBRID_DEMO",
                    "search_scorer"     : "rsf",
                    "search_fusion"     : "VECTOR_ONLY",
                    "vector": {
                        "search_text"  : "root cause analysis of electrical failures",
                        "search_mode"  : "DOCUMENT",
                        "aggregator"   : "MAX",
                        "score_weight" : 1
                    },
                    "return": {
                        "values" : ["rowid", "vector_score", "chunk_text"],
                        "topN"   : 5
                    }
                }')
        ) FROM DUAL
    """)

    result = cursor.fetchone()[0]
    print("=== Vector-Only Search ===")
    print_json(result)

    ok("Query complete.")
else:
    print("Optional hybrid section skipped: set RUN_HYBRID_SECTION = True in the first hybrid cell to run vector-only hybrid search.")


### Text-only search

Now a pure keyword search for "Substation Gamma" through the same hybrid index.

In [ ]:
if RUN_HYBRID_SECTION:
    # Pure text-only (keyword) search via the hybrid index
    cursor.execute("""
        SELECT DBMS_HYBRID_VECTOR.SEARCH(
                JSON('{
                    "hybrid_index_name" : "IDX_HYBRID_DEMO",
                    "search_scorer"     : "rsf",
                    "search_fusion"     : "TEXT_ONLY",
                    "text": {
                        "contains"     : "Substation AND Gamma",
                        "score_weight" : 1
                    },
                    "return": {
                        "values" : ["rowid", "text_score", "chunk_text"],
                        "topN"   : 5
                    }
                }')
        ) FROM DUAL
    """)

    result = cursor.fetchone()[0]
    print("=== Text-Only Search ===")
    print_json(result)

    ok("Query complete.")
else:
    ok("Optional hybrid section skipped: set RUN_HYBRID_SECTION = True in the first hybrid cell to run text-only hybrid search.")


### Hybrid search: Best of both worlds

Now we combine both searches. The hybrid index fuses the keyword scores and semantic scores using Relative Score Fusion (RSF), giving us results that are both semantically relevant AND contain the exact terms we need.

In [ ]:
if RUN_HYBRID_SECTION:
    # Full hybrid search: semantic + keyword combined
    cursor.execute("""
        SELECT DBMS_HYBRID_VECTOR.SEARCH(
                JSON('{
                    "hybrid_index_name" : "IDX_HYBRID_DEMO",
                    "search_scorer"     : "rsf",
                    "search_fusion"     : "UNION",
                    "vector": {
                        "search_text"  : "root cause analysis of electrical failures",
                        "search_mode"  : "DOCUMENT",
                        "aggregator"   : "MAX",
                        "score_weight" : 1
                    },
                    "text": {
                        "contains"     : "Substation AND Gamma",
                        "score_weight" : 1
                    },
                    "return": {
                        "values" : ["rowid", "score", "vector_score", "text_score", "chunk_text"],
                        "topN"   : 5
                    }
                }')
        ) FROM DUAL
    """)

    result = cursor.fetchone()[0]
    result = rename_fused_score(result)
    print("=== Hybrid Search (UNION with RSF) ===")
    print_json(result)

    ok("Query complete.")
else:
    ok("Optional hybrid section skipped: set RUN_HYBRID_SECTION = True in the first hybrid cell to run combined hybrid search.")


> ✅ **Checkpoint: Hybrid search is working**
>
> You combined semantic similarity with keyword search over the same maintenance content.
>
> **Expected result:** rows include vector and/or text scores, letting you compare meaning-based retrieval with exact term matching.


### Why is the text_score column in some rows 0?

You probably noticed that some rows in the UNION results has a `text_score` of 0. That's not a bug.

With `UNION` fusion, the result set includes rows from *either* the vector search OR the text search. The top 5 by combined score are all coming purely from the vector side because "root cause analysis of electrical failures" is semantically rich and scores high. Those particular rows don't contain the exact words "Substation" and "Gamma" in their content, so their `text_score` is 0. The text matches that *do* mention "Substation Gamma" are getting pushed below the top 5 because their vector relevance to "electrical failures" is lower.

You also may have noticed that `fused_score` and `vector_score` are different even when `text_score` is 0. That's because RSF normalizes the raw scores before combining them. `vector_score` is the raw semantic similarity, while `fused_score` is the normalized, fused result.

The fix: change the fusion to `INTERSECT`. INTERSECT only returns rows that appear in **both** the text search AND the vector search. That's the real power of hybrid search: find results that are semantically about electrical failures AND mention Substation Gamma by name.

### Hybrid search with INTERSECT fusion

In [ ]:
if RUN_HYBRID_SECTION:
    # Hybrid search with INTERSECT: only rows matching BOTH searches
    cursor.execute("""
        SELECT DBMS_HYBRID_VECTOR.SEARCH(
                JSON('{
                    "hybrid_index_name" : "IDX_HYBRID_DEMO",
                    "search_scorer"     : "rsf",
                    "search_fusion"     : "INTERSECT",
                    "vector": {
                        "search_text"  : "root cause analysis of electrical failures",
                        "search_mode"  : "DOCUMENT",
                        "aggregator"   : "MAX",
                        "score_weight" : 1
                    },
                    "text": {
                        "contains"     : "Substation AND Gamma",
                        "score_weight" : 1
                    },
                    "return": {
                        "values" : ["rowid", "score", "vector_score", "text_score", "chunk_text"],
                        "topN"   : 5
                    }
                }')
        ) FROM DUAL
    """)

    result = cursor.fetchone()[0]
    result = rename_fused_score(result)
    print("=== Hybrid Search (INTERSECT with RSF) ===")
    print_json(result)

    ok("Query complete.")
else:
    ok("Optional hybrid section skipped: set RUN_HYBRID_SECTION = True in the first hybrid cell to run INTERSECT hybrid search.")


Now every row has both a non-zero `vector_score` AND a non-zero `text_score`. These are results that are semantically relevant to electrical failures and contain the words "Substation Gamma."

### Tuning the balance: Weighting text higher

RSF lets you control how much influence each search type has on the final score. By increasing `score_weight` on the text side, we tell Oracle to favor results where the keyword match is strong. This is useful when exact terminology matters more than broad semantic similarity.

In [ ]:
if RUN_HYBRID_SECTION:
    # Hybrid search with heavier text weighting
    cursor.execute("""
        SELECT DBMS_HYBRID_VECTOR.SEARCH(
                JSON('{
                    "hybrid_index_name" : "IDX_HYBRID_DEMO",
                    "search_scorer"     : "rsf",
                    "search_fusion"     : "UNION",
                    "vector": {
                        "search_text"  : "root cause analysis of electrical failures",
                        "search_mode"  : "DOCUMENT",
                        "aggregator"   : "MAX",
                        "score_weight" : 1
                    },
                    "text": {
                        "contains"     : "Substation OR Gamma",
                        "score_weight" : 5
                    },
                    "return": {
                        "values" : ["rowid", "score", "vector_score", "text_score", "chunk_text"],
                        "topN"   : 5
                    }
                }')
        ) FROM DUAL
    """)

    result = cursor.fetchone()[0]
    result = rename_fused_score(result)
    print("=== Hybrid Search (UNION, text weight = 5) ===")
    print_json(result)

    ok("Query complete.")
else:
    print("Optional hybrid section skipped: set RUN_HYBRID_SECTION = True in the first hybrid cell to run weighted hybrid search.")


With `score_weight` set to 5 on the text side (versus 1 on the vector side), results containing "Substation" or "Gamma" get a significant scoring boost. Compare the `text_score` and `vector_score` columns to see how the weighting shifts which results rise to the top. This is one of the "knobs" you can tune for your application without changing any code.

### Interpreting the comparison

- **Vector-only** found semantically relevant electrical failure content, but may have included results from other assets entirely.
- **Text-only** found documents mentioning "Substation Gamma" by name, but missed semantically related content that described the same issues using different wording.
- **Hybrid** found the best of both: results that are semantically relevant to electrical failures AND associated with Substation Gamma. The fused score reflects both relevance dimensions.

This is exactly the scenario from the presentation: a query like *"What safety incidents involved asset CMP-2241 and what were the root causes?"* needs both types of search working together.

In [ ]:
if RUN_HYBRID_SECTION:
    # ╭─ EXERCISE §6.A ─────────────────────────────────────────────────────╮
    # │ OPTIONAL TASK: tune hybrid search scoring.                          │
    # │                                                                     │
    # │ HOW TO COMPLETE THIS CELL:                                          │
    # │   1. Change TEXT_WEIGHT and VECTOR_WEIGHT.                          │
    # │   2. Compare which rows move up or down.                            │
    # │   3. Set RUN_EXERCISE = True and run the cell.                      │
    # │                                                                     │
    # │ Stuck? Expand the 🔎 solution cell below and paste the full answer. │
    # ╰─────────────────────────────────────────────────────────────────────╯

    RUN_EXERCISE = False
    VECTOR_WEIGHT = 1
    TEXT_WEIGHT = 1

    if RUN_EXERCISE:
        hybrid_query = f"""
            SELECT DBMS_HYBRID_VECTOR.SEARCH(
                    JSON('{{
                        "hybrid_index_name" : "IDX_HYBRID_DEMO",
                        "search_scorer"     : "rsf",
                        "search_fusion"     : "UNION",
                        "vector": {{
                            "search_text"  : "root cause analysis of electrical failures",
                            "search_mode"  : "DOCUMENT",
                            "aggregator"   : "MAX",
                            "score_weight" : {VECTOR_WEIGHT}
                        }},
                        "text": {{
                            "contains"     : "Substation OR Gamma",
                            "score_weight" : {TEXT_WEIGHT}
                        }},
                        "return": {{
                            "values" : ["rowid", "score", "vector_score", "text_score", "chunk_text"],
                            "topN"   : 5
                        }}
                    }}')
            ) FROM DUAL
        """
        cursor.execute(hybrid_query)
        result = rename_fused_score(cursor.fetchone()[0])
        print_json(result)
    else:
        print("Optional exercise skipped. Change weights, then set RUN_EXERCISE = True to run it.")

    ok("Query complete.")
else:
    ok("Optional hybrid section skipped: set RUN_HYBRID_SECTION = True in the first hybrid cell to run hybrid scoring exercise.")


<details>
<summary>🔎 <b>Reveal solution: §6.A hybrid scoring weights</b></summary>

```python
RUN_EXERCISE = True
VECTOR_WEIGHT = 1
TEXT_WEIGHT = 5
```

Increase `TEXT_WEIGHT` when exact terminology matters. Increase `VECTOR_WEIGHT` when semantic similarity matters more than exact words.

</details>


### Cleanup

In [ ]:
if RUN_HYBRID_SECTION:
    # Clean up the hybrid search demo objects defensively.
    safe_execute_ddl("DROP INDEX idx_hybrid_demo FORCE", "Drop IDX_HYBRID_DEMO")
    safe_execute_ddl("DROP TABLE hybrid_search_demo PURGE", "Drop HYBRID_SEARCH_DEMO")
    safe_execute_ddl("BEGIN DBMS_VECTOR_CHAIN.DROP_PREFERENCE('prism_hybrid_pref'); END;", "Drop PRISM_HYBRID_PREF")
    conn.commit()
    print("Hybrid search demo cleanup complete.")
    ok("Drop complete.")
else:
    ok("Optional hybrid cleanup skipped because RUN_HYBRID_SECTION is False.")


> ✅ **Checkpoint: Data Fundamentals lab complete**
>
> You explored the same Prism dataset through relational, JSON, graph, vector, and optional hybrid search patterns.
>
> **Takeaway:** Oracle can support AI application data access patterns in one database without copying data into separate purpose-built stores.


---
## Section 7: Summary and next steps

In this notebook you:

- **Connected** to Oracle AI Database 26ai and explored the Prism smart city dataset across relational tables, JSON columns, helper views, a property graph, and vector embeddings
- **Inserted** a new maintenance log and watched the vectorization pipeline create chunks and embeddings using an in-database ONNX model
- **Inspected** the HNSW vector index and performed semantic search across all content types
- **Combined** relational JOINs, criticality scoring, JSON dot notation, graph traversal, and vector search in a single SQL query

All of this happened in **one database, with one query language (SQL), and zero synchronization overhead**.

### Further Reading

- [Chunking Strategies](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/perform-chunking-embedding-and-vector-generation.html)
- [HNSW Vector Indexes](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/manage-different-categories-vector-indexes.html)
- [Oracle AI Vector Search Documentation](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/)

In [ ]:
# Close the database connection
for name in ["cursor", "conn"]:
    obj = globals().get(name)
    if obj is None:
        continue
    try:
        obj.close()
    except Exception as exc:
        print(f"{name} already closed or could not be closed cleanly: {exc}")

ok("Connection cleanup complete. Lab complete!")
